# MathProver: Find or certify proofs for theorems

This notebook demonstrates the complete first version of MathProver. A language model may suggest a proof, but **Lean 4 is the verifier**: a result is certified only when Lean accepts the generated formal proof.

Run the cells from top to bottom. Before starting, activate the `mathprover` Python environment, install the project with `pip install -e .`, and select that environment as this notebook's kernel. The project also needs a working `lean_project/` directory with Mathlib installed.

There are two APIs:

- `certify(source)` checks Lean code that *you* supply.
- `prove(theorem_statement)` asks OpenAI for a proof and has Lean check it.

For `certify`, include `import Mathlib` and a complete proof. For `prove`, provide only a Lean theorem declaration—do not include `:= by`, a proof, or `import Mathlib`.

## 1. Certify a proof we supply

The next cell passes a complete Lean file to the local Lean compiler. `simp` proves that adding zero to a natural number changes nothing. A `True` result means Lean checked every part of the theorem and proof.

A successful Lean invocation commonly produces no diagnostic text; that is expected.

In [2]:
from mathprover import certify

source = """
import Mathlib

theorem add_zero_test (n : Nat) : n + 0 = n := by
  simp
"""

result = certify(source)

print("Certified:", result.certified)
print(result.stderr)

Certified: True



## 2. Confirm that Lean rejects a false claim

Certification is meaningful only if invalid proofs fail. The next cell attempts to prove `1 = 2` using `rfl`. Lean rejects the source, so `Certified` must be `False`; the error message is Lean's explanation.

This is why model output alone is never treated as a mathematical proof.

In [3]:
bad_source = """
import Mathlib

theorem false_claim : 1 = 2 := by
  rfl
"""

result = certify(bad_source)

print("Certified:", result.certified)
print(result.stderr)

Certified: False



## 3. Generate a proof, then certify it

The next cell supplies only a Lean theorem statement. MathProver loads `OPENAI_API_KEY` from the top-level `.env` file, requests a Lean proof beginning with `by`, adds the Mathlib import, and calls Lean locally. If Lean rejects a candidate, its diagnostic is supplied to the model on the next attempt.

The default is at most three attempts. Model calls can take noticeably longer than Lean checking. For a quick experiment, use `max_attempts=1`.

The input is formal Lean, not natural language. A future formalization step can translate an English claim into a theorem statement, but it should show the formalized statement for human review before proving it.

In [1]:
from mathprover import prove

result = prove("""theorem alg_formula_1_test (a b : Nat) : (a + b)*(a - b) = a*a - b*b""",
              max_attempts = 2,
              verbose = 1)

print("Certified:", result.certified)
print("Attempts:", result.attempts)
print("Proof:")
print(result.proof)
print(result.lean_result.stderr)

[tactics] Trying 16 standard tactics in one compile...
[tactics] None closed the goal.
[grounding] 12 of 12 proposed names exist.
[attempt 1/2] Requesting 4 candidate(s) from gpt-5...
[attempt 1/2] Checking candidate 1/4 with Lean...
[attempt 1/2] Lean rejected candidate 1.
[attempt 1/2] Checking candidate 2/4 with Lean...
[attempt 1/2] Lean certified the proof.
Certified: True
Attempts: 1
Proof:
rcases Nat.le_total b a with hba | hab
· have H1 : (a + b) * (a - b) + b * b = a * a := by
    calc
      (a + b) * (a - b) + b * b
          = (a * (a - b) + b * (a - b)) + b * b := by
              simpa [Nat.add_mul]
      _ = a * (a - b) + (b * (a - b) + b * b) := by
              simpa [Nat.add_assoc]
      _ = a * (a - b) + b * ((a - b) + b) := by
              simpa [Nat.mul_add, Nat.add_assoc]
      _ = a * (a - b) + b * a := by
              simpa [Nat.sub_add_cancel hba]
      _ = a * (a - b) + a * b := by
              simpa [Nat.mul_comm]
      _ = a * ((a - b) + b) := by
         

## What to try next

Change the theorem statement to another small result from natural-number algebra. Keep the theorem declaration syntactically complete, but omit `:= by`. Record whether it certifies, how many attempts it used, and Lean's diagnostic if it fails.